# DATA LOADING PIPELINE (POLARS)

# EXPONENTIAL STATE-SPACE MODEL - V5

Advanced exponential modeling pipeline with hierarchical state-space approach

**Purpose:**
- Load raw train/test data
- Apply comprehensive missing value imputation
- Build hierarchical exponential state-space models
- Generate ensemble predictions with uncertainty quantification

**Pipeline Architecture:**
1. **Data Loading & Preprocessing** - Raw data → Clean features
2. **MCMC Imputation** - Advanced Bayesian imputation for critical features
3. **State-Space Construction** - Dimension reduction + state vectors
4. **Hierarchical A-Matrix Estimation** - Multi-level dynamics modeling
5. **Exponential Forecasting** - State transition predictions
6. **Uncertainty Quantification** - Bayesian + GARCH modeling
7. **Ensemble Integration** - Combine with LightGBM v4

**Expected Performance:**
- Better long-term forecasting (H10, H25)
- Improved uncertainty handling
- Hierarchical structure for complex dynamics

## 1.1 IMPORTS AND CONFIGURATION

In [2]:
# ============================================
# IMPORTS AND CONFIGURATION
# ============================================
import polars as pl
from pathlib import Path
import time
import numpy as np

# Polars configuration
pl.Config.set_tbl_rows(20)
pl.Config.set_tbl_cols(20)

# Set up paths
base_dir = Path("..")
train_path = base_dir / "data" / "processed" / "train_processed_v1.parquet"  # Float32 without clipping
# Test data options - choose one:
# Option 1: Original test data (Float64)
#test_path = base_dir / "data" / "test.parquet"
# Option 2: Processed test data (Float32) - uncomment to use
test_path = base_dir / "data" / "processed" / "test_processed_v1.parquet"
processed_dir = base_dir / "data" / "processed"
results_dir = base_dir / "Results"
print("✅ Configuration complete")
print(f"📁 Base directory: {base_dir}")

✅ Configuration complete
📁 Base directory: ..


## 1.2 DATA LOADING

In [3]:
# ============================================
# LOAD DATASETS
# ============================================

print("="*60)
print("LOADING DATASETS")
print("="*60)

train_full = pl.read_parquet(train_path)
test_full = pl.read_parquet(test_path)

print(f"✅ Train loaded: {train_full.shape}")
print(f"✅ Test loaded: {test_full.shape}")

# szybki sanity check
print("\n📊 Dtypes (train):")
print(train_full.dtypes[:10])

LOADING DATASETS
✅ Train loaded: (5337414, 94)
✅ Test loaded: (1447107, 92)

📊 Dtypes (train):
[String, Categorical, Categorical, Categorical, Int16, Int16, Float32, Float32, Float32, Float32]


## 1.3 FEATURE GROUPS

In [4]:
# ============================================
# FEATURE GROUPS (PYTHONIC + CORRECT LOGIC)
# ============================================

FEATURE_GROUPS = {
    "ultra": ['feature_bz', 'feature_aw', 'feature_cc'],
    "high": ['feature_az', 'feature_bl', 'feature_l', 'feature_m'],
    "test38": ['feature_w', 'feature_x', 'feature_y', 'feature_z'],
    "core": ['feature_at', 'feature_by', 'feature_ay', 'feature_cd',
             'feature_ce', 'feature_cf', 'feature_al'],
}

# all feature columns
all_features = [c for c in train_full.columns if c.startswith("feature_")]

# detect missing features
missing_cols = [
    c for c in all_features
    if train_full[c].null_count() > 0
]

no_missing_cols = [
    c for c in all_features
    if train_full[c].null_count() == 0
]

# flatten defined groups
high_priority_features = [
    f for group in FEATURE_GROUPS.values() for f in group
]

# 🔥 FIX: LOW only from missing columns
priority_low = [
    f for f in missing_cols if f not in high_priority_features
]

# ============================================
# PRINT SUMMARY
# ============================================

print("🎯 Feature groups summary:")
for k, v in FEATURE_GROUPS.items():
    print(f"   {k.upper():<7} ({len(v)}): {v[:3]}{'...' if len(v)>3 else ''}")

print(f"\n📊 Total features: {len(all_features)}")
print(f"📉 With NaN: {len(missing_cols)}")
print(f"✅ Without NaN: {len(no_missing_cols)}")
print(f"🧊 LOW priority (only NaN features!): {len(priority_low)}")

🎯 Feature groups summary:
   ULTRA   (3): ['feature_bz', 'feature_aw', 'feature_cc']
   HIGH    (4): ['feature_az', 'feature_bl', 'feature_l']...
   TEST38  (4): ['feature_w', 'feature_x', 'feature_y']...
   CORE    (7): ['feature_at', 'feature_by', 'feature_ay']...

📊 Total features: 86
📉 With NaN: 48
✅ Without NaN: 38
🧊 LOW priority (only NaN features!): 30


## 2.2 FILL TEST38

In [5]:
# ============================================
# FILL TEST38 (NO LEAKAGE - GROUPED FORWARD FILL)
# ============================================

TEST38 = FEATURE_GROUPS["test38"]

# Grouped forward fill
train_full = train_full.with_columns([
    pl.col(c).forward_fill().over(['code', 'sub_code', 'sub_category']).alias(c)
    for c in TEST38
])

test_full = test_full.with_columns([
    pl.col(c).forward_fill().over(['code', 'sub_code', 'sub_category']).alias(c)
    for c in TEST38
    if c in test_full.columns
])

# Clean remaining NaN (first rows in groups)
for c in TEST38:
    if train_full[c].null_count() > 0:
        train_full = train_full.with_columns([
            pl.col(c).fill_null(1e-8).alias(c)
        ])
    if c in test_full.columns and test_full[c].null_count() > 0:
        test_full = test_full.with_columns([
            pl.col(c).fill_null(1e-8).alias(c)
        ])

print(f"✅ TEST38 forward-filled + zeros: {len(TEST38)}")

✅ TEST38 forward-filled + zeros: 4


## 2.3 FILL CORE + LOW

In [6]:
# ============================================
# FILL CORE + LOW (NO LEAKAGE - GROUPED FORWARD FILL)
# ============================================

CORE = FEATURE_GROUPS["core"]
LOW = priority_low
fill_cols = CORE + LOW

# Grouped forward fill by categorical keys
train_full = train_full.with_columns([
    pl.col(c).forward_fill().over(['code', 'sub_code', 'sub_category']).alias(c)
    for c in fill_cols
])

test_full = test_full.with_columns([
    pl.col(c).forward_fill().over(['code', 'sub_code', 'sub_category']).alias(c)
    for c in fill_cols
    if c in test_full.columns
])

print(f"✅ CORE + LOW forward-filled: {len(fill_cols)}")

# ============================================
# CLEAN REMAINING NaN IN THESE COLUMNS (fill with 0)
# ============================================

for c in fill_cols:
    if train_full[c].null_count() > 0:
        train_full = train_full.with_columns([
            pl.col(c).fill_null(1e-8).alias(c)
        ])
    if c in test_full.columns and test_full[c].null_count() > 0:
        test_full = test_full.with_columns([
            pl.col(c).fill_null(1e-8).alias(c)
        ])

print(f"✅ Remaining NaN in CORE+LOW filled with 0")

✅ CORE + LOW forward-filled: 37
✅ Remaining NaN in CORE+LOW filled with 0


## 2.5 ULTRA/HIGH FEATURES

**ULTRA Priority:** Δw 10x + test 5% missing - **Bayesian MCMC ready**

**HIGH Priority:** Δy=18 impact - **Bayesian MCMC ready**

In [7]:
# ============================================
# MCMC IMPUTATION WITH SAVED MODELS (SAMPLED)
# ============================================
print("="*60)
print("BAYESIAN IMPUTATION - ULTRA & HIGH (SAMPLED DATA)")
print("="*60)

from sklearn.linear_model import BayesianRidge
import numpy as np
import joblib
from pathlib import Path

ULTRA = ['feature_bz', 'feature_aw', 'feature_cc']
HIGH = ['feature_az', 'feature_bl', 'feature_l', 'feature_m']
MCMC_FEATURES = ULTRA + HIGH

model_dir = Path("../models")
model_dir.mkdir(parents=True, exist_ok=True)

# ============================================
# STEP 1: TRAIN MODELS ON SAMPLED TRAIN DATA
# ============================================
print("\n📊 STEP 1: Training Bayesian models on sampled train data...")

models = {}

for feat in MCMC_FEATURES:
    print(f"  Training model for: {feat}")

    # Get rows where feature is not null
    train_clean = train_full.filter(pl.col(feat).is_not_null())

    if len(train_clean) < 100:
        print(f"    ⚠️  Not enough data → forward fill")
        models[feat] = None
        continue

    # 🔥 SAMPLE to reduce memory (use 500k rows max)
    sample_size = min(500_000, len(train_clean))
    train_sample = train_clean.sample(n=sample_size, seed=42)

    print(f"    Using {len(train_sample):,} rows (sampled from {len(train_clean):,})")

    # Select predictors: other features (without target, without weight)
    pred_cols = [c for c in train_full.columns
                 if c.startswith('feature_')
                 and c != feat
                 and c not in MCMC_FEATURES]

    # Add frequency encodings for categories
    for cat in ['code', 'sub_code', 'sub_category']:
        freq_col = f'{cat}_freq'
        freq_map = train_sample.group_by(cat).agg(pl.len().alias(freq_col)).to_dict(as_series=False)
        train_sample = train_sample.with_columns([
            pl.col(cat).replace_strict(freq_map[cat], freq_map[freq_col], default=0).alias(freq_col)
        ])
        pred_cols.append(freq_col)

    # Add temporal features
    train_sample = train_sample.with_columns([
        (pl.col('ts_index') / 3600).alias('ts_norm'),
        (pl.col('horizon') / 25).alias('horizon_norm')
    ])
    pred_cols.extend(['ts_norm', 'horizon_norm'])

    # Prepare data
    X_train = train_sample.select(pred_cols).to_numpy()
    y_train = train_sample.select(feat).to_numpy().ravel()

    # Handle potential NaNs
    if np.any(np.isnan(X_train)):
        X_train = np.nan_to_num(X_train, nan=0.0)

    # Train Bayesian Ridge
    if feat in HIGH:
        # Higher precision for HIGH (Δy=18)
        model = BayesianRidge(max_iter=300, tol=1e-5, alpha_1=1e-5, lambda_1=1e-5)
    else:
        # Standard for ULTRA
        model = BayesianRidge(max_iter=200, tol=1e-4, alpha_1=1e-6, lambda_1=1e-6)

    model.fit(X_train, y_train)

    # Save model and predictors
    models[feat] = {
        'model': model,
        'pred_cols': pred_cols,
        'is_high': feat in HIGH
    }

    # Save to disk
    joblib.dump(model, model_dir / f"{feat}_model.pkl")
    print(f"    ✅ Model saved")

# ============================================
# STEP 2: APPLY TO TEST
# ============================================
print("\n📊 STEP 2: Applying Bayesian imputation to test data...")

for feat in MCMC_FEATURES:
    print(f"  Imputing: {feat}")

    if models[feat] is None:
        test_full = test_full.with_columns([
            pl.col(feat).forward_fill().over(['code', 'sub_code', 'sub_category']).alias(feat)
        ])
        continue

    null_mask = test_full[feat].is_null()

    if null_mask.sum() == 0:
        print(f"    ✅ No nulls in test")
        continue

    # Prepare predictors for test
    test_temp = test_full.clone()

    for cat in ['code', 'sub_code', 'sub_category']:
        freq_col = f'{cat}_freq'
        if freq_col not in test_temp.columns:
            train_freq = train_full.group_by(cat).agg(pl.len().alias(freq_col))
            test_temp = test_temp.join(train_freq, on=cat, how='left')
            test_temp = test_temp.with_columns(pl.col(freq_col).fill_null(0))

    test_temp = test_temp.with_columns([
        (pl.col('ts_index') / 3600).alias('ts_norm'),
        (pl.col('horizon') / 25).alias('horizon_norm')
    ])

    pred_cols = models[feat]['pred_cols']
    X_test = test_temp.filter(null_mask).select(pred_cols).to_numpy()

    if np.any(np.isnan(X_test)):
        X_test = np.nan_to_num(X_test, nan=0.0)

    y_pred = models[feat]['model'].predict(X_test)

    # Create DataFrame with predictions
    null_ids = test_temp.filter(null_mask).select('id').to_numpy().ravel()
    pred_df = pl.DataFrame({
        'id': null_ids,
        f'{feat}_pred': y_pred
    })

    # Join and fill nulls
    test_full = test_full.join(pred_df, on='id', how='left')
    test_full = test_full.with_columns([
        pl.col(feat).fill_null(pl.col(f'{feat}_pred')).alias(feat)
    ])
    test_full = test_full.drop(f'{feat}_pred')

    print(f"    ✅ Imputed {null_mask.sum():,} rows")

# ============================================
# STEP 3: CLEANUP
# ============================================
print("\n📊 STEP 3: Cleaning up temporary columns...")

temp_cols = [c for c in test_full.columns if c.endswith('_freq') or c in ['ts_norm', 'horizon_norm']]
test_full = test_full.drop(temp_cols)

print(f"✅ Bayesian imputation complete for {len(MCMC_FEATURES)} features")
print(f"   ULTRA (3): {ULTRA}")
print(f"   HIGH (4): {HIGH}")

BAYESIAN IMPUTATION - ULTRA & HIGH (SAMPLED DATA)

📊 STEP 1: Training Bayesian models on sampled train data...
  Training model for: feature_bz
    Using 500,000 rows (sampled from 5,185,692)
    ✅ Model saved
  Training model for: feature_aw
    Using 500,000 rows (sampled from 5,132,220)
    ✅ Model saved
  Training model for: feature_cc
    Using 500,000 rows (sampled from 5,334,779)
    ✅ Model saved
  Training model for: feature_az
    Using 500,000 rows (sampled from 5,326,257)
    ✅ Model saved
  Training model for: feature_bl
    Using 500,000 rows (sampled from 5,326,257)
    ✅ Model saved
  Training model for: feature_l
    Using 500,000 rows (sampled from 5,336,114)
    ✅ Model saved
  Training model for: feature_m
    Using 500,000 rows (sampled from 5,299,244)
    ✅ Model saved

📊 STEP 2: Applying Bayesian imputation to test data...
  Imputing: feature_bz
    ✅ Imputed 54,635 rows
  Imputing: feature_aw
    ✅ Imputed 55,028 rows
  Imputing: feature_cc
    ✅ Imputed 1,564 r

In [8]:
# Po MCMC – ostateczne czyszczenie
feature_cols_all = [c for c in train_full.columns if c.startswith('feature_')]
nan_cols = [c for c in feature_cols_all if train_full[c].null_count() > 0]

if nan_cols:
    print(f"⚠️ Po MCMC nadal {len(nan_cols)} kolumn z NaN, wypełniam małą liczbą (1e-8):")
    for c in nan_cols:
        print(f"   {c}: {train_full[c].null_count():,} NaN")
        train_full = train_full.with_columns([
            pl.col(c).fill_null(1e-8).alias(c)
        ])
    print("✅ NaN wypełnione 1e-8")
else:
    print("✅ Po MCMC brak NaN – można robić PCA")

⚠️ Po MCMC nadal 7 kolumn z NaN, wypełniam małą liczbą (1e-8):
   feature_l: 1,300 NaN
   feature_m: 38,170 NaN
   feature_aw: 205,194 NaN
   feature_az: 11,157 NaN
   feature_bl: 11,157 NaN
   feature_bz: 151,722 NaN
   feature_cc: 2,635 NaN
✅ NaN wypełnione 1e-8


## 2.6 SAVE PROCESSED DATA

In [11]:
# ============================================
# SAVE PROCESSED DATA - V4 (MCMC IMPUTATION)
# ============================================
print("="*60)
print("SAVING PROCESSED DATA - V4 MCMC PIPELINE")
print("="*60)

# Create directory
processed_dir = base_dir / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

# Define paths
train_out_path = processed_dir / "train_processed_v5.parquet"
test_out_path = processed_dir / "test_processed_v5.parquet"

print("💾 Saving enhanced datasets...")

# Save files
train_full.write_parquet(train_out_path)
test_full.write_parquet(test_out_path)

print(f"✅ Train saved: {train_out_path}")
print(f"✅ Test saved: {test_out_path}")

# Get file sizes
train_size = train_out_path.stat().st_size / 1024**2
test_size = test_out_path.stat().st_size / 1024**2

print(f"\n📁 Files created:")
print(f"   Train: {train_size:.1f} MB")
print(f"   Test: {test_size:.1f} MB")

# Summary for v4
feature_cols_count = len([c for c in train_full.columns if c.startswith('feature_')])
print(f"\n🎯 V4 MCMC Pipeline Complete!")
print(f"\n📈 Pipeline Summary:")
print(f"   Total columns: {train_full.shape[1]}")
print(f"   Feature columns: {feature_cols_count}")
print(f"   Imputation methods:")
print(f"      • Bayesian MCMC (ULTRA + HIGH): {len(ULTRA) + len(HIGH)} features")
print(f"      • Forward fill (TEST38, CORE, LOW): {len(TEST38) + len(CORE) + len(LOW)} features")
print(f"\n🚀 v4 ready for LightGBM training!")
print(f"💡 Expected: Better score than v3")

SAVING PROCESSED DATA - V4 MCMC PIPELINE
💾 Saving enhanced datasets...
✅ Train saved: ..\data\processed\train_processed_v5.parquet
✅ Test saved: ..\data\processed\test_processed_v5.parquet

📁 Files created:
   Train: 414.2 MB
   Test: 69.9 MB

🎯 V4 MCMC Pipeline Complete!

📈 Pipeline Summary:
   Total columns: 94
   Feature columns: 86
   Imputation methods:
      • Bayesian MCMC (ULTRA + HIGH): 7 features
      • Forward fill (TEST38, CORE, LOW): 41 features

🚀 v4 ready for LightGBM training!
💡 Expected: Better score than v3


In [9]:
# Przed PCA: sprawdź, czy są jeszcze jakieś NaN
nan_cols = [col for col in train_full.columns if train_full[col].null_count() > 0]
print(f"Kolumny z NaN przed PCA: {len(nan_cols)}")
if nan_cols:
    print(nan_cols[:20])  # pokaż pierwsze 20
    # Opcjonalnie: zobacz ile NaN jest w pierwszej z nich
    col = nan_cols[0]
    print(f"W kolumnie {col}: {train_full[col].null_count():,} NaN")

Kolumny z NaN przed PCA: 0


In [10]:
# ============================================
# DIAGNOSTYKA: WSZYSTKIE NaN W TEŚCIE
# ============================================
print("Sprawdzam wszystkie kolumny w test_full:")

all_cols = test_full.columns
cols_with_nan = []

for col in all_cols:
    null_count = test_full[col].null_count()
    if null_count > 0:
        cols_with_nan.append((col, null_count))
        print(f"  {col}: {null_count:,} NaN")

if cols_with_nan:
    print(f"\n⚠️ Znaleziono {len(cols_with_nan)} kolumn z NaN w teście:")
    for col, cnt in cols_with_nan[:20]:
        print(f"   - {col}: {cnt:,}")
else:
    print("✅ Brak NaN w test_full")

Sprawdzam wszystkie kolumny w test_full:
  feature_am: 881 NaN
  feature_an: 881 NaN
  feature_ao: 881 NaN
  feature_ap: 881 NaN
  feature_aq: 972 NaN
  feature_ar: 972 NaN
  feature_as: 972 NaN
  feature_ba: 972 NaN
  feature_bb: 972 NaN
  feature_bc: 972 NaN
  feature_bd: 881 NaN
  feature_be: 881 NaN
  feature_bf: 881 NaN
  feature_bg: 881 NaN
  feature_bh: 881 NaN
  feature_bk: 881 NaN
  feature_bm: 881 NaN
  feature_bn: 881 NaN
  feature_bo: 881 NaN
  feature_bp: 972 NaN
  feature_bq: 972 NaN
  feature_br: 972 NaN
  feature_bs: 881 NaN
  feature_bt: 881 NaN
  feature_bu: 881 NaN
  feature_bv: 881 NaN
  feature_bw: 881 NaN
  feature_bx: 881 NaN
  feature_cb: 972 NaN

⚠️ Znaleziono 29 kolumn z NaN w teście:
   - feature_am: 881
   - feature_an: 881
   - feature_ao: 881
   - feature_ap: 881
   - feature_aq: 972
   - feature_ar: 972
   - feature_as: 972
   - feature_ba: 972
   - feature_bb: 972
   - feature_bc: 972
   - feature_bd: 881
   - feature_be: 881
   - feature_bf: 881
   - fe

In [11]:
# ============================================
# FINAL CLEANUP: TEST – FORWARD FILL, potem 1e-8
# ============================================
print("Ostateczne czyszczenie testu przed PCA...")

# Najpierw forward fill po grupach (bezpieczne, używa przeszłości)
test_full = test_full.with_columns([
    pl.col(col).forward_fill().over(['code', 'sub_code', 'sub_category']).alias(col)
    for col in test_full.columns
    if test_full[col].null_count() > 0
])

# Sprawdź co zostało
remaining_cols = [col for col in test_full.columns if test_full[col].null_count() > 0]

if remaining_cols:
    print(f"Po forward fill nadal {len(remaining_cols)} kolumn z NaN:")
    for col in remaining_cols[:10]:
        print(f"  {col}: {test_full[col].null_count():,} NaN")

    # Dla pozostałych – wypełnij 1e-8 (bezpieczna mała liczba)
    for col in remaining_cols:
        test_full = test_full.with_columns(
            pl.col(col).fill_null(1e-8).alias(col)
        )
    print("✅ Pozostałe NaN wypełnione 1e-8")
else:
    print("✅ Po forward fill brak NaN")

# Finalna kontrola
total_nan = sum(test_full[col].null_count() for col in test_full.columns)
print(f"Total NaN w test po czyszczeniu: {total_nan}")

Ostateczne czyszczenie testu przed PCA...
✅ Po forward fill brak NaN
Total NaN w test po czyszczeniu: 0


## 4.1 DIMENSIONALITY REDUCTION (SVD)

**Purpose:** Reduce 134 features → 50 components for numerical stability

**Implementation:**
- Fit SVD on training data
- Transform both train and test
- Create state vectors: [y_target, svd_1, ..., svd_50]

In [12]:
# ============================================
# STEP 5: SVD REDUCTION (134 → 50)
# ============================================
print("="*60)
print("STEP 5: SVD DIMENSIONALITY REDUCTION")
print("="*60)

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# Get all feature columns
feature_cols = [c for c in train_full.columns if c.startswith('feature_')]
print(f"Original features: {len(feature_cols)}")

# Prepare data for SVD
X_train_full = train_full.select(feature_cols).to_numpy()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_train_full)

# PCA to 50 components (more stable than SVD directly)
pca = PCA(n_components=50, random_state=42)
X_pca = pca.fit_transform(X_scaled)

print(f"Reduced to: {X_pca.shape[1]} components")
print(f"Explained variance: {pca.explained_variance_ratio_.sum():.4f}")

# Create PCA features for train
pca_cols = [f'pca_{i}' for i in range(50)]
train_full = train_full.with_columns([
    pl.Series(name=pca_cols[i], values=X_pca[:, i]) for i in range(50)
])

# For test
X_test_full = test_full.select(feature_cols).to_numpy()
X_test_scaled = scaler.transform(X_test_full)
X_test_pca = pca.transform(X_test_scaled)

test_full = test_full.with_columns([
    pl.Series(name=pca_cols[i], values=X_test_pca[:, i]) for i in range(50)
])

print(f"✅ PCA features added to train and test")

STEP 5: SVD DIMENSIONALITY REDUCTION
Original features: 86
Reduced to: 50 components
Explained variance: 0.9256
✅ PCA features added to train and test


fast diagnosis

In [15]:
# Sprawdź ile grup ma >=5 wierszy
group_counts = train_full.group_by(['code', 'sub_code', 'sub_category', 'horizon']).agg(
    pl.len().alias('n')
)
print(f"Grupy z n>=1: {group_counts.filter(pl.col('n') >= 1).height}")
print(f"Grupy z n>=3: {group_counts.filter(pl.col('n') >=5).height}")
print(f"Grupy z n>=20: {group_counts.filter(pl.col('n') >= 20).height}")
print(f"Total grup: {group_counts.height}")

Grupy z n>=1: 36923
Grupy z n>=3: 36747
Grupy z n>=20: 36093
Total grup: 36923


## 6 ESTIMATE A-MATRIX PER GROUP

**Purpose:** Estimate transition matrix A for each hierarchical level

**Implementation:**
- For each combination of (code, sub_code, sub_category, horizon)
- Build X = state vector
- Estimate A from regression: ΔX = A * X
- Solution: X(t+h) = e^(A * h) * X(t)

In [16]:
# ============================================
# STEP 6: ESTIMATE MATRIX A PER GROUP
# ============================================
print("="*60)
print("STEP 6: ESTIMATING STATE TRANSITION MATRIX A")
print("="*60)

from scipy.linalg import expm
from sklearn.linear_model import LinearRegression

# State vector: y_target + PCA components
state_cols = ['y_target'] + pca_cols
print(f"State vector size: {len(state_cols)}")

# Store models per group
A_matrices = {}

# Group by code, sub_code, sub_category, horizon
groups = train_full.group_by(['code', 'sub_code', 'sub_category', 'horizon'])

for (code, sub, cat, h), group in groups:
    group = group.sort('ts_index')
    if len(group) < 5:  # Not enough data
        continue

    # Build state matrix X (rows = time steps)
    X_state = group.select(state_cols).to_numpy()

    # Compute ΔX = X_{t+1} - X_t
    X_t = X_state[:-1]
    X_t1 = X_state[1:]
    dX = X_t1 - X_t

    # Estimate A: dX = A * X_t
    reg = LinearRegression(fit_intercept=False)
    reg.fit(X_t, dX)
    A = reg.coef_.T  # dX = A @ X_t

    # Store
for (code, sub, cat, h), group in groups:
    group = group.sort('ts_index')
    if len(group) < 5:
        continue

    # Buduj key
    key = f"{code}_{sub}_{cat}_{h}"

    # Build state matrix X
    X_state = group.select(state_cols).to_numpy()

    # ... reszta kodu ...

    A_matrices[key] = {
        'A': A,
        'last_state': X_state[-1],
        'horizon': h,
        'code': code,
        'sub': sub,
        'cat': cat
    }

    if len(A_matrices) % 100 == 0:
        print(f"  Processed {len(A_matrices)} groups")

print(f"✅ Estimated A matrices for {len(A_matrices)} groups")

STEP 6: ESTIMATING STATE TRANSITION MATRIX A
State vector size: 51
  Processed 100 groups
  Processed 200 groups
  Processed 300 groups
  Processed 400 groups
  Processed 500 groups
  Processed 600 groups
  Processed 700 groups
  Processed 800 groups
  Processed 900 groups
  Processed 1000 groups
  Processed 1100 groups
  Processed 1200 groups
  Processed 1300 groups
  Processed 1400 groups
  Processed 1500 groups
  Processed 1600 groups
  Processed 1700 groups
  Processed 1800 groups
  Processed 1900 groups
  Processed 2000 groups
  Processed 2100 groups
  Processed 2200 groups
  Processed 2300 groups
  Processed 2400 groups
  Processed 2500 groups
  Processed 2600 groups
  Processed 2700 groups
  Processed 2800 groups
  Processed 2900 groups
  Processed 3000 groups
  Processed 3100 groups
  Processed 3200 groups
  Processed 3300 groups
  Processed 3400 groups
  Processed 3500 groups
  Processed 3600 groups
  Processed 3700 groups
  Processed 3800 groups
  Processed 3900 groups
  Proc

In [17]:
# Sprawdź czy A_matrices nie ma błędów
print(f"Liczba grup: {len(A_matrices)}")
print(f"Przykładowy klucz: {list(A_matrices.keys())[0]}")
print(f"Kształt A: {A_matrices[list(A_matrices.keys())[0]]['A'].shape}")
print(f"Przykładowe wartości w A: {A_matrices[list(A_matrices.keys())[0]]['A'][0, :5]}")

Liczba grup: 36747
Przykładowy klucz: K8I5QG74_SE7KARJH_DPPUO5X2_25
Kształt A: (51, 51)
Przykładowe wartości w A: [-0.45907645 -0.1365331   0.06780496 -0.07069549  0.55807999]


## 7 HIERARCHICAL A-MATRIX

**Purpose:** Combine hierarchical A matrices with weighted averaging

**Implementation:**
- A_code – dynamics at code level
- A_sub – dynamics at sub_code level
- A_cat – dynamics at category level
- Weighted average: A_final = α*A_code + β*A_sub + γ*A_cat

In [18]:
# ============================================
# STEP 7: HIERARCHICAL MATRIX A (CODE → SUB → CAT)
# ============================================
print("="*60)
print("STEP 7: BUILDING HIERARCHICAL STATE TRANSITION")
print("="*60)

# ============================================
# 7.1 Aggregate A matrices by hierarchy level
# ============================================

# Store A matrices per level
A_by_code = {}
A_by_sub = {}
A_by_cat = {}

for key, data in A_matrices.items():
    parts = key.split('_')
    code = parts[0]
    sub = parts[1]
    cat = parts[2]
    horizon = int(parts[3])

    A = data['A']

    # Aggregate by code
    if code not in A_by_code:
        A_by_code[code] = []
    A_by_code[code].append(A)

    # Aggregate by sub_code
    sub_key = f"{code}_{sub}"
    if sub_key not in A_by_sub:
        A_by_sub[sub_key] = []
    A_by_sub[sub_key].append(A)

    # Aggregate by category
    if cat not in A_by_cat:
        A_by_cat[cat] = []
    A_by_cat[cat].append(A)

# Compute mean A per level
print("Computing mean A per hierarchy level...")

A_code_mean = {}
for code, matrices in A_by_code.items():
    A_code_mean[code] = np.mean(matrices, axis=0)
    print(f"  Code {code}: {len(matrices)} matrices → mean A")

A_sub_mean = {}
for sub_key, matrices in A_by_sub.items():
    A_sub_mean[sub_key] = np.mean(matrices, axis=0)

A_cat_mean = {}
for cat, matrices in A_by_cat.items():
    A_cat_mean[cat] = np.mean(matrices, axis=0)
    print(f"  Category {cat}: {len(matrices)} matrices → mean A")

print(f"✅ Hierarchical A matrices computed:")
print(f"   Code level: {len(A_code_mean)} unique codes")
print(f"   Sub-code level: {len(A_sub_mean)} unique sub-codes")
print(f"   Category level: {len(A_cat_mean)} unique categories")

# ============================================
# 7.2 Weighted combination for each group
# ============================================

# Hyperparameters for hierarchical weights (tunable)
# Higher weight for more specific level
ALPHA_CODE = 0.3   # weight for code-level dynamics
ALPHA_SUB = 0.5    # weight for sub-code level (most specific)
ALPHA_CAT = 0.2    # weight for category level

print(f"\nHierarchy weights: code={ALPHA_CODE}, sub={ALPHA_SUB}, cat={ALPHA_CAT}")

# Store combined A for each group
A_combined = {}

for key, data in A_matrices.items():
    parts = key.split('_')
    code = parts[0]
    sub = parts[1]
    cat = parts[2]
    horizon = int(parts[3])

    sub_key = f"{code}_{sub}"

    # Get A from each level (fallback to global if not available)
    A_code = A_code_mean.get(code, np.zeros_like(data['A']))
    A_sub = A_sub_mean.get(sub_key, np.zeros_like(data['A']))
    A_cat = A_cat_mean.get(cat, np.zeros_like(data['A']))

    # Weighted combination
    A_combined[key] = ALPHA_CODE * A_code + ALPHA_SUB * A_sub + ALPHA_CAT * A_cat

print(f"✅ Combined hierarchical A for {len(A_combined)} groups")

# ============================================
# 7.3 Verify stability of combined A matrices
# ============================================

print("\nChecking stability of combined A matrices...")

def check_stability(A):
    """Check if all eigenvalues have negative real parts"""
    eigvals = np.linalg.eigvals(A)
    return np.all(np.real(eigvals) < 1e-6)  # stable if real parts < 0

stable_count = 0
unstable_count = 0

for key, A in A_combined.items():
    if check_stability(A):
        stable_count += 1
    else:
        unstable_count += 1
        if unstable_count <= 5:
            eigvals = np.linalg.eigvals(A)
            print(f"  ⚠️ Unstable A for {key}: max real part = {np.max(np.real(eigvals)):.4f}")

print(f"\nStability summary:")
print(f"  Stable matrices: {stable_count}")
print(f"  Unstable matrices: {unstable_count}")

if unstable_count > 0:
    print(f"\n⚠️ Warning: {unstable_count} groups have unstable dynamics")
    print("   This may cause exploding predictions. Consider:")
    print("   - Increasing regularization when estimating A")
    print("   - Using spectral normalization")
    print("   - Clipping eigenvalues > 0")

# ============================================
# 7.4 Store combined A for prediction
# ============================================

# Save A matrices for test prediction
A_for_prediction = {}

for key, data in A_matrices.items():
    parts = key.split('_')
    code = parts[0]
    sub = parts[1]
    cat = parts[2]
    horizon = int(parts[3])

    A_comb = A_combined[key]
    last_state = data['last_state']

    A_for_prediction[key] = {
        'A': A_comb,
        'last_state': last_state,
        'horizon': horizon,
        'code': code,
        'sub': sub,
        'cat': cat
    }

print(f"\n✅ Hierarchical A ready for prediction on {len(A_for_prediction)} groups")

STEP 7: BUILDING HIERARCHICAL STATE TRANSITION
Computing mean A per hierarchy level...
  Code K8I5QG74: 1460 matrices → mean A
  Code CXEQN6KB: 1400 matrices → mean A
  Code 6LB028J8: 2580 matrices → mean A
  Code W4S29LF4: 1440 matrices → mean A
  Code MLAAMU3K: 1140 matrices → mean A
  Code 83EG83KQ: 2056 matrices → mean A
  Code OSJL3A7Y: 3510 matrices → mean A
  Code VFWIFJPS: 2055 matrices → mean A
  Code X9BZ68VQ: 1360 matrices → mean A
  Code 2RBMUWP1: 960 matrices → mean A
  Code 10BAVIDU: 920 matrices → mean A
  Code 4KUR2ZOZ: 1440 matrices → mean A
  Code 1HEMHZK2: 1755 matrices → mean A
  Code K7Y1TTAH: 1755 matrices → mean A
  Code MRV5UON2: 1440 matrices → mean A
  Code 84J8BJFZ: 1440 matrices → mean A
  Code 660DZME0: 2260 matrices → mean A
  Code SJZP0OVU: 2260 matrices → mean A
  Code W2MW3G2L: 2020 matrices → mean A
  Code WH61ASEA: 781 matrices → mean A
  Code HYOGKLEV: 800 matrices → mean A
  Code QAQDDTPJ: 775 matrices → mean A
  Code EP12UF2K: 1140 matrices → mean 

## 8 BAYESIAN UNCERTAINTY

**Purpose:** Instead of single A matrix, sample from parameter distribution

**Implementation:**
- Use Bayesian Ridge to estimate parameter distributions
- Generate multiple A matrix samples
- Quantify prediction uncertainty

In [19]:
# ============================================
# STEP 8: BAYESIAN UNCERTAINTY FOR λ (Wariant B)
# ============================================
print("="*60)
print("STEP 8: BAYESIAN UNCERTAINTY FOR DOMINANT λ")
print("="*60)

# ============================================
# 8.1 Extract dominant λ from A matrices
# ============================================

print("\nExtracting dominant eigenvalues from A matrices...")

lambda_by_group = {}

for key, data in A_matrices.items():
    A = data['A']

    # Compute eigenvalues
    eigvals = np.linalg.eigvals(A)

    # Dominant eigenvalue (largest real part)
    dominant_lambda = np.max(np.real(eigvals))

    # Also store spectral radius for stability info
    spectral_radius = np.max(np.abs(eigvals))

    lambda_by_group[key] = {
        'dominant': dominant_lambda,
        'spectral_radius': spectral_radius,
        'code': data['code'],
        'sub': data['sub'],
        'cat': data['cat'],
        'horizon': data['horizon'],
        'last_state': data['last_state']  # keep for later
    }

print(f"✅ Extracted dominant λ for {len(lambda_by_group)} groups")

# ============================================
# 8.2 Statistics of λ
# ============================================

all_lambdas = [data['dominant'] for data in lambda_by_group.values()]
print(f"\nλ statistics:")
print(f"  Mean: {np.mean(all_lambdas):.6f}")
print(f"  Std: {np.std(all_lambdas):.6f}")
print(f"  Min: {np.min(all_lambdas):.6f}")
print(f"  Max: {np.max(all_lambdas):.6f}")
print(f"  Positive λ (>0): {sum(1 for l in all_lambdas if l > 0)} / {len(all_lambdas)}")

# ============================================
# 8.3 Bayesian estimation per hierarchy level
# ============================================

print("\nEstimating Bayesian posterior for λ per level...")

# Collect λ by hierarchy level
lambda_by_code = {}
lambda_by_sub = {}
lambda_by_cat = {}

for key, data in lambda_by_group.items():
    code = data['code']
    sub = data['sub']
    cat = data['cat']
    lam = data['dominant']

    if code not in lambda_by_code:
        lambda_by_code[code] = []
    lambda_by_code[code].append(lam)

    sub_key = f"{code}_{sub}"
    if sub_key not in lambda_by_sub:
        lambda_by_sub[sub_key] = []
    lambda_by_sub[sub_key].append(lam)

    if cat not in lambda_by_cat:
        lambda_by_cat[cat] = []
    lambda_by_cat[cat].append(lam)

def estimate_bayesian_lambda(values):
    """Estimate posterior λ ~ Normal(μ, σ)"""
    if len(values) < 3:
        return np.mean(values), np.std(values)

    # Simple empirical Bayes
    mu = np.mean(values)
    sigma = np.std(values)

    return mu, sigma

# Compute Bayesian λ for each level
bayesian_lambda = {
    'code': {},
    'sub': {},
    'cat': {}
}

print("  Code level:")
for code, lambdas in lambda_by_code.items():
    mu, sigma = estimate_bayesian_lambda(lambdas)
    bayesian_lambda['code'][code] = {'mu': mu, 'sigma': sigma, 'n': len(lambdas)}
    print(f"    {code}: μ={mu:.4f}, σ={sigma:.4f}, n={len(lambdas)}")

print("  Category level:")
for cat, lambdas in lambda_by_cat.items():
    mu, sigma = estimate_bayesian_lambda(lambdas)
    bayesian_lambda['cat'][cat] = {'mu': mu, 'sigma': sigma, 'n': len(lambdas)}
    print(f"    {cat}: μ={mu:.4f}, σ={sigma:.4f}, n={len(lambdas)}")

print(f"  Sub-code level: {len(lambda_by_sub)} groups")

# ============================================
# 8.4 Sample λ from posterior for prediction
# ============================================

print("\nPreparing posterior sampling for predictions...")

def sample_lambda_posterior(code, sub, cat, n_samples=100):
    """Sample λ from hierarchical posterior"""
    # Get distributions for each level
    code_dist = bayesian_lambda['code'].get(code, {'mu': 0.0, 'sigma': 0.1})
    sub_key = f"{code}_{sub}"
    sub_dist = bayesian_lambda['sub'].get(sub_key, {'mu': 0.0, 'sigma': 0.1})
    cat_dist = bayesian_lambda['cat'].get(cat, {'mu': 0.0, 'sigma': 0.1})

    # Sample from each level
    samples_code = np.random.normal(code_dist['mu'], code_dist['sigma'], n_samples)
    samples_sub = np.random.normal(sub_dist['mu'], sub_dist['sigma'], n_samples)
    samples_cat = np.random.normal(cat_dist['mu'], cat_dist['sigma'], n_samples)

    # Hierarchical weighting
    weights = [0.3, 0.5, 0.2]  # code, sub, cat
    samples_combined = weights[0] * samples_code + weights[1] * samples_sub + weights[2] * samples_cat

    return samples_combined

# Store λ samples for each group
lambda_samples_by_group = {}

for key, data in lambda_by_group.items():
    code = data['code']
    sub = data['sub']
    cat = data['cat']

    samples = sample_lambda_posterior(code, sub, cat, n_samples=200)

    lambda_samples_by_group[key] = {
        'samples': samples,
        'mu': np.mean(samples),
        'sigma': np.std(samples),
        'dominant_raw': data['dominant'],
        'code': code,
        'sub': sub,
        'cat': cat,
        'last_state': data['last_state'],
        'horizon': data['horizon']
    }

print(f"✅ Posterior samples prepared for {len(lambda_samples_by_group)} groups")

# ============================================
# 8.5 Uncertainty metrics
# ============================================

print("\nComputing uncertainty metrics...")

cv_list = []
for key, data in lambda_samples_by_group.items():
    cv = data['sigma'] / (abs(data['mu']) + 1e-8)
    cv_list.append(cv)

print(f"  Average CV: {np.mean(cv_list):.4f}")
print(f"  Max CV: {np.max(cv_list):.4f}")
print(f"  Min CV: {np.min(cv_list):.4f}")

print(f"\n✅ Bayesian λ ready for forecasting")

STEP 8: BAYESIAN UNCERTAINTY FOR DOMINANT λ

Extracting dominant eigenvalues from A matrices...
✅ Extracted dominant λ for 36747 groups

λ statistics:
  Mean: 0.302496
  Std: 0.000000
  Min: 0.302496
  Max: 0.302496
  Positive λ (>0): 36747 / 36747

Estimating Bayesian posterior for λ per level...
  Code level:
    K8I5QG74: μ=0.3025, σ=0.0000, n=1460
    CXEQN6KB: μ=0.3025, σ=0.0000, n=1400
    6LB028J8: μ=0.3025, σ=0.0000, n=2580
    W4S29LF4: μ=0.3025, σ=0.0000, n=1440
    MLAAMU3K: μ=0.3025, σ=0.0000, n=1140
    83EG83KQ: μ=0.3025, σ=0.0000, n=2056
    OSJL3A7Y: μ=0.3025, σ=0.0000, n=3510
    VFWIFJPS: μ=0.3025, σ=0.0000, n=2055
    X9BZ68VQ: μ=0.3025, σ=0.0000, n=1360
    2RBMUWP1: μ=0.3025, σ=0.0000, n=960
    10BAVIDU: μ=0.3025, σ=0.0000, n=920
    4KUR2ZOZ: μ=0.3025, σ=0.0000, n=1440
    1HEMHZK2: μ=0.3025, σ=0.0000, n=1755
    K7Y1TTAH: μ=0.3025, σ=0.0000, n=1755
    MRV5UON2: μ=0.3025, σ=0.0000, n=1440
    84J8BJFZ: μ=0.3025, σ=0.0000, n=1440
    660DZME0: μ=0.3025, σ=0.0000,

## 9 EXPONENTIAL FORECASTING

**Purpose:** Generate predictions using exponential state transitions

**Implementation:**
- For test data: X_pred = e^(A * h) * X_last
- Extract y_pred from state vector
- Use hierarchical A matrices with uncertainty

In [22]:
# ============================================
# STEP 9: PROGNOZA WYKŁADNICZA (PROSTA I DZIAŁAJĄCA)
# ============================================
print("="*60)
print("STEP 9: EXPONENTIAL FORECAST (SIMPLE VERSION)")
print("="*60)

# 1. Oblicz średnie λ dla każdej kombinacji (sub_category, horizon)
lambda_by_cat_h = {}
for key, data in lambda_samples_by_group.items():
    parts = key.split('_')
    cat = parts[2]
    h = int(parts[3])
    cat_h_key = f"{cat}_{h}"
    if cat_h_key not in lambda_by_cat_h:
        lambda_by_cat_h[cat_h_key] = []
    lambda_by_cat_h[cat_h_key].append(data['mu'])

lambda_cat_h_avg = {k: np.mean(v) for k, v in lambda_by_cat_h.items()}
global_lambda = np.mean([data['mu'] for data in lambda_samples_by_group.values()])

print(f"Znaleziono λ dla {len(lambda_cat_h_avg)} kombinacji (sub_category, horizon)")
print(f"Global λ: {global_lambda:.6f}")

# 2. Oblicz ostatnią wartość y_target dla każdej kombinacji (sub_category, horizon) z train
y_last_map = {}
for (cat, h), group in train_full.group_by(['sub_category', 'horizon']):
    group_sorted = group.sort('ts_index')
    last_y = group_sorted.select('y_target').to_numpy()[-1][0]
    y_last_map[f"{cat}_{h}"] = last_y

print(f"Znaleziono y_last dla {len(y_last_map)} kombinacji")

# 3. Dla testu, utwórz kolumnę predykcji
test_full = test_full.with_columns([
    pl.lit(0.0).alias('exp_prediction')
])

# 4. Dla każdej kombinacji (sub_category, horizon) przypisz predykcję
for (cat, h), group in test_full.group_by(['sub_category', 'horizon']):
    key = f"{cat}_{h}"

    # Weź λ (jeśli nie ma dla tej kombinacji, użyj global)
    lam = lambda_cat_h_avg.get(key, global_lambda)

    # Weź y_last (jeśli nie ma, użyj 0)
    y_last = y_last_map.get(key, 0.0)

    # Prognoza
    pred = y_last * np.exp(lam * h)

    # Aktualizuj test_full
    mask = (test_full['sub_category'] == cat) & (test_full['horizon'] == h)
    test_full = test_full.with_columns(
        pl.when(mask).then(pred).otherwise(pl.col('exp_prediction')).alias('exp_prediction')
    )

print(f"✅ Predykcje przypisane")

# 5. Sprawdź wynik
preds = test_full.select('exp_prediction').to_numpy().ravel()
print(f"\nStatystyki predykcji:")
print(f"  Średnia: {np.mean(preds):.6f}")
print(f"  Std: {np.std(preds):.6f}")
print(f"  Min: {np.min(preds):.6f}")
print(f"  Max: {np.max(preds):.6f}")
print(f"  NaN: {np.isnan(preds).sum()}")

STEP 9: EXPONENTIAL FORECAST (SIMPLE VERSION)
Znaleziono λ dla 20 kombinacji (sub_category, horizon)
Global λ: 0.151244
Znaleziono y_last dla 20 kombinacji
✅ Predykcje przypisane

Statystyki predykcji:
  Średnia: 0.001488
  Std: 0.004205
  Min: -0.005837
  Max: 0.012038
  NaN: 0


In [23]:
# ============================================
# PODSUMOWANIE PO KROKU 9
# ============================================

print("="*60)
print("PODSUMOWANIE PROGNOZ WYKŁADNICZYCH")
print("="*60)

# Sprawdź czy kolumna exp_prediction istnieje
if 'exp_prediction' in test_full.columns:
    # Ile wierszy ma predykcję (nie-null)
    pred_count = test_full.filter(pl.col('exp_prediction').is_not_null()).height
    print(f"Wiersze z predykcją: {pred_count:,} / {len(test_full):,}")

    # Statystyki predykcji
    preds = test_full.select('exp_prediction').to_numpy().ravel()
    print(f"Średnia: {np.mean(preds):.6f}")
    print(f"Std: {np.std(preds):.6f}")
    print(f"Min: {np.min(preds):.6f}")
    print(f"Max: {np.max(preds):.6f}")

    # Sprawdź czy są NaN
    nan_count = test_full.filter(pl.col('exp_prediction').is_null()).height
    if nan_count > 0:
        print(f"⚠️ Nadal {nan_count:,} wierszy bez predykcji")
else:
    print("❌ Brak kolumny exp_prediction – Krok 9 nie wykonał się poprawnie")

PODSUMOWANIE PROGNOZ WYKŁADNICZYCH
Wiersze z predykcją: 1,447,107 / 1,447,107
Średnia: 0.001488
Std: 0.004205
Min: -0.005837
Max: 0.012038


In [25]:
# ============================================
# METRYKA KAGGLE
# ============================================

def _clip01(x: float) -> float:
    return float(np.minimum(np.maximum(x, 0.0), 1.0))

def weighted_rmse_score(y_target, y_pred, w) -> float:
    denom = np.sum(w * y_target ** 2)
    ratio = np.sum(w * (y_target - y_pred) ** 2) / denom
    clipped = _clip01(ratio)
    val = 1.0 - clipped
    return float(np.sqrt(val))

print("✅ Metric defined")
# ============================================
# V5 EXPONENTIAL MODEL - TRAIN SCORE
# ============================================

# Wygeneruj predykcje dla train (używając tej samej logiki)
train_exp_preds = []
for row in train_full.iter_rows(named=True):
    cat = row['sub_category']
    h = row['horizon']
    key = f"{cat}_{h}"
    lam = lambda_cat_h_avg.get(key, global_lambda)
    y_last = y_last_map.get(key, 0.0)
    pred = y_last * np.exp(lam * h)
    train_exp_preds.append(pred)

train_exp_preds = np.array(train_exp_preds)
y_true = train_full.select('y_target').to_numpy().ravel()
w = train_full.select('weight').to_numpy().ravel()

v5_score = weighted_rmse_score(y_true, train_exp_preds, w)

print(f"v5 Exponential train score: {v5_score:.6f}")
print(f"v4 LightGBM train score: 0.266488")
print(f"Różnica: {v5_score - 0.266488:.6f}")

✅ Metric defined
v5 Exponential train score: 0.000000
v4 LightGBM train score: 0.266488
Różnica: -0.266488


In [26]:
print("Pierwsze 5 porównań:")
for i in range(5):
    print(f"  Pred: {train_exp_preds[i]:.6f} | True: {y_true[i]:.6f} | Różnica: {train_exp_preds[i] - y_true[i]:.6f}")

Pierwsze 5 porównań:
  Pred: 0.011499 | True: -0.551324 | Różnica: 0.562823
  Pred: -0.000074 | True: -0.315583 | Różnica: 0.315509
  Pred: 0.000558 | True: -0.362894 | Różnica: 0.363452
  Pred: 0.000962 | True: -0.667023 | Różnica: 0.667985
  Pred: 0.011499 | True: -0.437398 | Różnica: 0.448897


## 10 GARCH ON RESIDUALS

**Purpose:** Model residual volatility and calibrate predictions

**Implementation:**
- Residual = y_true - y_pred (on train)
- Train GARCH(1,1)
- Calibrate predictions

In [ ]:
# ============================================
# STEP 10: GARCH VOLATILITY MODELLING
# ============================================
print("="*60)
print("STEP 10: GARCH(1,1) FOR RESIDUAL VOLATILITY")
print("="*60)

try:
    from arch import arch_model
    HAS_ARCH = True
    print("✅ arch package available")
except ImportError:
    HAS_ARCH = False
    print("⚠️ arch package not available. Install with: pip install arch")
    print("Skipping GARCH step...")

# ============================================
# 10.1 Compute residuals on train
# ============================================

if HAS_ARCH:
    print("\nComputing residuals on training data...")

    # Create predictions for train using exponential model
    train_with_exp = train_full.clone()
    train_with_exp = train_with_exp.with_columns([
        pl.lit(0.0).alias('exp_prediction_train')
    ])

    # For each group in train, compute exponential prediction
    train_groups = train_full.group_by(['code', 'sub_code', 'sub_category', 'horizon'])

    for (code, sub, cat, horizon), group in train_groups:
        group_key = f"{code}_{sub}_{cat}_{horizon}"

        if group_key not in forecasts_by_group:
            continue

        # Get λ samples for this group
        lambda_samples = lambda_samples_by_group[group_key]['samples']

        # For each row in group, compute prediction based on previous y
        group_sorted = group.sort('ts_index')
        y_values = group_sorted.select('y_target').to_numpy().ravel()

        predictions = []
        for i, y_current in enumerate(y_values):
            if i == 0:
                # First row: use group mean or forward fill
                pred = np.mean(y_values[:5]) if len(y_values) >= 5 else y_current
            else:
                # Predict next from previous: y_prev * exp(λ * horizon)
                # But here horizon is fixed per group
                y_prev = y_values[i-1]
                # Use mean λ for deterministic prediction
                lambda_mu = np.mean(lambda_samples)
                pred = y_prev * np.exp(lambda_mu * horizon)
            predictions.append(pred)

        predictions = np.array(predictions)

        # Update train_with_exp
        ids = group_sorted.select('id').to_numpy().ravel()
        for idx, row_id in enumerate(ids):
            train_with_exp = train_with_exp.with_columns([
                pl.when(pl.col('id') == row_id)
                .then(pl.lit(predictions[idx]))
                .otherwise(pl.col('exp_prediction_train'))
                .alias('exp_prediction_train')
            ])

    # Compute residuals
    train_with_exp = train_with_exp.with_columns([
        (pl.col('y_target') - pl.col('exp_prediction_train')).alias('residual')
    ])

    print(f"✅ Residuals computed for {len(train_with_exp):,} rows")

    # ============================================
    # 10.2 Train GARCH per horizon
    # ============================================

    print("\nTraining GARCH(1,1) models per horizon...")

    garch_models = {}
    garch_forecasts = {}

    for horizon in [1, 3, 10, 25]:
        print(f"\n  Horizon {horizon}:")

        # Get residuals for this horizon
        residuals_h = train_with_exp.filter(pl.col('horizon') == horizon).select('residual').to_numpy().ravel()

        # Remove zeros and extreme values for stability
        residuals_h = residuals_h[~np.isnan(residuals_h)]
        residuals_h = residuals_h[np.abs(residuals_h) < np.percentile(np.abs(residuals_h), 99)]

        if len(residuals_h) < 100:
            print(f"    ⚠️ Not enough data ({len(residuals_h)}), skipping")
            garch_models[horizon] = None
            continue

        try:
            # Fit GARCH(1,1)
            model = arch_model(residuals_h * 100, vol='Garch', p=1, q=1, dist='normal')
            garch_fit = model.fit(disp='off')

            garch_models[horizon] = garch_fit

            # Forecast volatility for test
            forecast = garch_fit.forecast(horizon=1)
            volatility_forecast = np.sqrt(forecast.variance.values[-1, 0]) / 100  # rescale back

            garch_forecasts[horizon] = {
                'volatility': volatility_forecast,
                'params': garch_fit.params
            }

            print(f"    ✅ GARCH trained")
            print(f"       Omega: {garch_fit.params['omega']:.6f}")
            print(f"       Alpha: {garch_fit.params['alpha[1]']:.6f}")
            print(f"       Beta: {garch_fit.params['beta[1]']:.6f}")
            print(f"       Forecast volatility: {volatility_forecast:.6f}")

        except Exception as e:
            print(f"    ❌ GARCH failed: {e}")
            garch_models[horizon] = None

    # ============================================
    # 10.3 Apply GARCH calibration to predictions
    # ============================================

    print("\nApplying GARCH calibration to test predictions...")

    # Copy test predictions
    test_full = test_full.with_columns([
        pl.col('exp_prediction').alias('exp_prediction_calibrated')
    ])

    for horizon in [1, 3, 10, 25]:
        if garch_models[horizon] is None:
            continue

        # Get volatility forecast
        vol_forecast = garch_forecasts[horizon]['volatility']

        # Adjust predictions: add volatility component
        # Idea: y_calibrated = y_exp * (1 + α * volatility)
        # Higher volatility = wider prediction range
        alpha = 0.1  # tunable parameter

        mask = test_full['horizon'] == horizon

        test_full = test_full.with_columns([
            pl.when(mask)
            .then(pl.col('exp_prediction') * (1 + alpha * vol_forecast))
            .otherwise(pl.col('exp_prediction_calibrated'))
            .alias('exp_prediction_calibrated')
        ])

        print(f"  Horizon {horizon}: volatility adjustment = {alpha * vol_forecast:.6f}")

    print(f"✅ GARCH calibration applied")

else:
    print("\n⚠️ GARCH step skipped (arch package not available)")
    # Copy original predictions as fallback
    test_full = test_full.with_columns([
        pl.col('exp_prediction').alias('exp_prediction_calibrated')
    ])

# ============================================
# 10.4 Summary
# ============================================

print("\n" + "="*60)
print("GARCH CALIBRATION SUMMARY")
print("="*60)

for horizon in [1, 3, 10, 25]:
    if HAS_ARCH and garch_models.get(horizon) is not None:
        original = test_full.filter(pl.col('horizon') == horizon).select('exp_prediction').to_numpy().ravel()
        calibrated = test_full.filter(pl.col('horizon') == horizon).select('exp_prediction_calibrated').to_numpy().ravel()

        print(f"\nHorizon {horizon}:")
        print(f"  Original mean: {np.mean(original):.6f}")
        print(f"  Calibrated mean: {np.mean(calibrated):.6f}")
        print(f"  Change: {np.mean(calibrated) - np.mean(original):.6f}")
    else:
        print(f"\nHorizon {horizon}: GARCH not applied")

print(f"\n✅ Step 10 complete")

## 10.1 ENSEMBLE WITH LIGHTGBM V4

**Purpose:** Combine exponential model with LightGBM v4 predictions

**Implementation:**
- Load LightGBM v4 predictions
- Optimal ensemble weights: y_final = w1*y_exp + w2*y_lgb
- Weight optimization based on validation performance

In [ ]:
# ============================================
# PLACEHOLDER: ENSEMBLE WITH LIGHTGBM v4
# ============================================
print("="*60)
print("ENSEMBLE PLACEHOLDER - TO BE COMPLETED AFTER v4 SUBMISSION")
print("="*60)

# ============================================
# Metric (zgodny z Kaggle)
# ============================================

def _clip01(x: float) -> float:
    return float(np.minimum(np.maximum(x, 0.0), 1.0))

def weighted_rmse_score(y_target, y_pred, w) -> float:
    """
    Official Kaggle metric for this competition
    """
    denom = np.sum(w * y_target ** 2)
    ratio = np.sum(w * (y_target - y_pred) ** 2) / denom
    clipped = _clip01(ratio)
    val = 1.0 - clipped
    return float(np.sqrt(val))

print("✅ Metric function ready")

# ============================================
# Ensemble weights (to be tuned)
# ============================================

# Suggested weights based on local performance
# v4 (LightGBM) vs v5 (Exponential)
WEIGHT_LGB = 0.5
WEIGHT_EXP = 0.5

print(f"\n📊 Ensemble configuration:")
print(f"   LightGBM v4 weight: {WEIGHT_LGB}")
print(f"   Exponential v5 weight: {WEIGHT_EXP}")

# ============================================
# How to ensemble (pseudocode)
# ============================================

print("\n🔧 How to combine predictions:")

print("""
1. Load predictions from v4:
   lgb_submission = pl.read_csv('lightgbm_benchmark_v4.csv')

2. Load predictions from v5:
   exp_submission = pl.read_csv('exponential_model_v5.csv')

3. Merge on id:
   ensemble = lgb_submission.join(exp_submission, on='id', suffix='_exp')

4. Weighted average:
   ensemble = ensemble.with_columns([
       (WEIGHT_LGB * pl.col('prediction') +
        WEIGHT_EXP * pl.col('prediction_exp')).alias('prediction_ensemble')
   ])

5. Save:
   ensemble.select(['id', 'prediction_ensemble']).write_csv('ensemble_v6.csv')
""")

# ============================================
# Validation ensemble (if both models have validation predictions)
# ============================================

print("\n📈 To find optimal weights:")

print("""
1. Use validation split (ts_index > 3500)
2. Compute predictions from both models on validation
3. Grid search over weights (0.0 to 1.0 step 0.1)
4. Select weights that minimize weighted_rmse_score
5. Apply same weights to test predictions

Example grid search:
best_score = float('inf')
best_weight = 0.5

for w in [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]:
    y_ensemble = w * y_lgb + (1-w) * y_exp
    score = weighted_rmse_score(y_true, y_ensemble, w_val)
    if score < best_score:
        best_score = score
        best_weight = w

print(f"Best weight: {best_weight}")
""")

# ============================================
# Note for future
# ============================================

print("\n" + "="*60)
print("💡 ENSEMBLE READY TO IMPLEMENT")
print("="*60)
print("""
Next steps:
1. Submit v5 (exponential model) to Kaggle
2. Submit v4 (LightGBM) to Kaggle
3. Compare public scores
4. If both are good, implement ensemble
5. Submit v6 (ensemble)

Expected: Ensemble > individual models
""")

print("\n✅ Placeholder added. Ensemble code ready when you need it.")

## 11.2 SAVE SUBMISSION

In [ ]:
# ============================================
# STEP 11: SUBMISSION FROM EXPONENTIAL MODEL
# ============================================
print("="*60)
print("STEP 11: GENERATING SUBMISSION")
print("="*60)

# ============================================
# 11.1 Prepare submission dataframe
# ============================================

# Use calibrated predictions (or fallback to original if GARCH skipped)
if 'exp_prediction_calibrated' in test_full.columns:
    submission_df = test_full.select(['id', 'exp_prediction_calibrated']).rename({'exp_prediction_calibrated': 'prediction'})
else:
    submission_df = test_full.select(['id', 'exp_prediction']).rename({'exp_prediction': 'prediction'})

print(f"Submission shape: {submission_df.shape}")
print("\nSample predictions:")
print(submission_df.head(10))

# ============================================
# 11.2 Basic statistics
# ============================================

print("\n" + "="*60)
print("PREDICTION STATISTICS")
print("="*60)

preds = submission_df.select('prediction').to_numpy().ravel()

print(f"Total predictions: {len(preds):,}")
print(f"Mean: {np.mean(preds):.6f}")
print(f"Std: {np.std(preds):.6f}")
print(f"Min: {np.min(preds):.6f}")
print(f"Max: {np.max(preds):.6f}")
print(f"Median: {np.median(preds):.6f}")

# By horizon
for horizon in [1, 3, 10, 25]:
    horizon_ids = test_full.filter(pl.col('horizon') == horizon).select('id').to_numpy().ravel()
    horizon_preds = submission_df.filter(pl.col('id').is_in(horizon_ids)).select('prediction').to_numpy().ravel()

    print(f"\nHorizon {horizon}:")
    print(f"  Count: {len(horizon_preds):,}")
    print(f"  Mean: {np.mean(horizon_preds):.6f}")
    print(f"  Std: {np.std(horizon_preds):.6f}")
    print(f"  Range: [{np.min(horizon_preds):.4f}, {np.max(horizon_preds):.4f}]")

# ============================================
# 11.3 Save submission
# ============================================

import datetime
from pathlib import Path

# Create results directory if needed
results_dir = base_dir / "Results"
results_dir.mkdir(parents=True, exist_ok=True)

# Generate filename
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
submission_filename = f"exponential_model_v5_{timestamp}.csv"
submission_path = results_dir / submission_filename

# Save
submission_df.write_csv(submission_path)

print(f"\n" + "="*60)
print("SUBMISSION SAVED")
print("="*60)
print(f"File: {submission_path}")
print(f"Size: {submission_path.stat().st_size / 1024**2:.2f} MB")
print(f"Rows: {len(submission_df):,}")

# ============================================
# 11.4 Optional: Validation on train split
# ============================================

print("\n" + "="*60)
print("VALIDATION ON TRAIN (ts_index > 3500)")
print("="*60)

# Split train into fit (ts_index <= 3500) and val (ts_index > 3500)
train_fit = train_full.filter(pl.col('ts_index') <= 3500)
train_val = train_full.filter(pl.col('ts_index') > 3500)

if len(train_val) > 0:
    print(f"Validation rows: {len(train_val):,}")

    # We need to compute exponential predictions for validation
    # This would require running the same exponential model on train_val
    # For now, just note that this is possible
    print("Note: To compute validation score, run exponential model on train_val")
    print("      using the same λ distributions from train_fit.")

    # Simplified validation using existing forecasts_by_group
    val_predictions = []
    val_actuals = []
    val_weights = []

    for row in train_val.iter_rows(named=True):
        group_key = f"{row['code']}_{row['sub_code']}_{row['sub_category']}_{row['horizon']}"

        if group_key in forecasts_by_group:
            # Use the mean prediction from the group
            pred = forecasts_by_group[group_key]['mean']
            val_predictions.append(pred)
            val_actuals.append(row['y_target'])
            val_weights.append(row['weight'])

    if len(val_predictions) > 0:
        val_predictions = np.array(val_predictions)
        val_actuals = np.array(val_actuals)
        val_weights = np.array(val_weights)

        val_score = weighted_rmse_score(val_actuals, val_predictions, val_weights)
        print(f"\nValidation Score (exponential model): {val_score:.6f}")
        print(f"Note: This is on {len(val_predictions):,} rows from train_val")
    else:
        print("Not enough validation predictions to compute score")
else:
    print("No validation rows (ts_index > 3500)")

print("\n" + "="*60)
print("✅ EXPONENTIAL MODEL v5 COMPLETE")
print("="*60)
print(f"\n🎯 Submission ready: {submission_filename}")
print(f"📊 Model type: Hierarchical Exponential with Bayesian λ + GARCH")
print(f"📈 Ready to upload to Kaggle!")